In [2]:
import pandas as pd
data=pd.read_json('News_Category_Dataset_v3.json',lines=True)
data=data[data['category'].isin(['POLITICS','SPORTS','TECHNOLOGY','WELLNESS'])]
data['text'] = data['headline'] + ' ' + data['short_description']
data['text']

17        Maury Wills, Base-Stealing Shortstop For Dodge...
21        Biden Says U.S. Forces Would Defend Taiwan If ...
24        ‘Beautiful And Sad At The Same Time’: Ukrainia...
26        Las Vegas Aces Win First WNBA Title, Chelsea G...
30        Biden Says Queen's Death Left 'Giant Hole' For...
                                ...                        
209479    This Is Only the Beginning: Surprising Advice ...
209523    Maria Sharapova Stunned By Victoria Azarenka I...
209524    Giants Over Patriots, Jets Over Colts Among  M...
209525    Aldon Smith Arrested: 49ers Linebacker Busted ...
209526    Dwight Howard Rips Teammates After Magic Loss ...
Name: text, Length: 58624, dtype: str

In [3]:
data

,link,headline,category,short_description,authors,date,text
17,https://www.huffpost.com/entry/dodgers-basebal...,"Maury Wills, Base-Stealing Shortstop For Dodge...",SPORTS,"Maury Wills, who helped the Los Angeles Dodger...","Beth Harris, AP",2022-09-20,"Maury Wills, Base-Stealing Shortstop For Dodge..."
21,https://www.huffpost.com/entry/biden-us-forces...,Biden Says U.S. Forces Would Defend Taiwan If ...,POLITICS,President issues vow as tensions with China rise.,,2022-09-19,Biden Says U.S. Forces Would Defend Taiwan If ...
24,https://www.huffpost.com/entry/ukraine-festiva...,‘Beautiful And Sad At The Same Time’: Ukrainia...,POLITICS,An annual celebration took on a different feel...,Jonathan Nicholson,2022-09-19,‘Beautiful And Sad At The Same Time’: Ukrainia...
26,https://www.huffpost.com/entry/2022-wnba-final...,"Las Vegas Aces Win First WNBA Title, Chelsea G...",SPORTS,Las Vegas never had a professional sports cham...,"Pat Eaton-Robb, AP",2022-09-19,"Las Vegas Aces Win First WNBA Title, Chelsea G..."
30,https://www.huffpost.com/entry/europe-britain-...,Biden Says Queen's Death Left 'Giant Hole' For...,POLITICS,"U.S. President Joe Biden, in London for the fu...","Darlene Superville, AP",2022-09-18,Biden Says Queen's Death Left 'Giant Hole' For...
...,...,...,...,...,...,...,...
209479,https://www.huffingtonpost.com/entry/life-tips...,This Is Only the Beginning: Surprising Advice ...,WELLNESS,"My great-aunt Ida loves to say, ""This is only ...","Ellie Knaus, Contributor\nAtomic Moms Podcast ...",2012-01-28,This Is Only the Beginning: Surprising Advice ...
209523,https://www.huffingtonpost.com/entry/maria-sha...,Maria Sharapova Stunned By Victoria Azarenka I...,SPORTS,"Afterward, Azarenka, more effusive with the pr...",,2012-01-28,Maria Sharapova Stunned By Victoria Azarenka I...
209524,https://www.huffingtonpost.com/entry/super-bow...,"Giants Over Patriots, Jets Over Colts Among M...",SPORTS,"Leading up to Super Bowl XLVI, the most talked...",,2012-01-28,"Giants Over Patriots, Jets Over Colts Among M..."
209525,https://www.huffingtonpost.com/entry/aldon-smi...,Aldon Smith Arrested: 49ers Linebacker Busted ...,SPORTS,CORRECTION: An earlier version of this story i...,,2012-01-28,Aldon Smith Arrested: 49ers Linebacker Busted ...


In [4]:
'''
token.lemma_.lower() — get root form and lowercase it
not token.is_stop — skip words like "the", "is", "by"
not token.is_punct — skip "!", ".", ","
len(token.lemma_) > 2 — skip very short words like "it", "us"
return ' '.join(tokens) - gives list into a string
Loops through every row in data['text']
str(text) — converts any NaN or non-string to string
re.sub(r'\s+', ' ', ...) — replaces multiple spaces/newlines with single space
Result is a plain list of cleaned strings
disable['ner','parser'] - just focus on lemmitization
nlp.pipe(clean_texts) — processes all texts in batches, much faster than one by one
disable=["ner", "parser"] — skips named entity recognition and parsing, not needed here, makes it faster
batch_size=512 — processes 512 texts at a time
preprocess(doc) — applies your function to each processed doc
Result stored in new column data['clean']
'''

'\ntoken.lemma_.lower() — get root form and lowercase it\nnot token.is_stop — skip words like "the", "is", "by"\nnot token.is_punct — skip "!", ".", ","\nlen(token.lemma_) > 2 — skip very short words like "it", "us"\nreturn \' \'.join(tokens) - gives list into a string\nLoops through every row in data[\'text\']\nstr(text) — converts any NaN or non-string to string\nre.sub(r\'\\s+\', \' \', ...) — replaces multiple spaces/newlines with single space\nResult is a plain list of cleaned strings\ndisable[\'ner\',\'parser\'] - just focus on lemmitization\nnlp.pipe(clean_texts) — processes all texts in batches, much faster than one by one\ndisable=["ner", "parser"] — skips named entity recognition and parsing, not needed here, makes it faster\nbatch_size=512 — processes 512 texts at a time\npreprocess(doc) — applies your function to each processed doc\nResult stored in new column data[\'clean\']\n'

### PREPROCESSING

In [8]:
import spacy,re

nlp = spacy.load("en_core_web_sm")

def preprocess(doc):
    tokens = [token.lemma_.lower() for token in doc 
              if not token.is_stop and not token.is_punct and len(token.lemma_) > 2]
    return ' '.join(tokens)

clean_texts = [re.sub(r'\s+', ' ', str(text)) for text in data['text']]

data['clean'] = [preprocess(doc) for doc in nlp.pipe(clean_texts, disable=["ner", "parser"], batch_size=512)]

print(data[['text', 'clean']].head())

                                                 text  \
17  Maury Wills, Base-Stealing Shortstop For Dodge...   
21  Biden Says U.S. Forces Would Defend Taiwan If ...   
24  ‘Beautiful And Sad At The Same Time’: Ukrainia...   
26  Las Vegas Aces Win First WNBA Title, Chelsea G...   
30  Biden Says Queen's Death Left 'Giant Hole' For...   

                                                clean  
17  maury wills base steal shortstop dodgers die m...  
21  biden say u.s. forces defend taiwan china inva...  
24  beautiful sad time ukrainian cultural festival...  
26  las vegas aces win wnba title chelsea gray nam...  
30  biden say queen death left giant hole royal fa...  


### Classification model

In [9]:
#tfidf + logistic regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X=data['clean']
y=data['category']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42) #straitfy to keep same as y to avoid imbalanced class

tfidf=TfidfVectorizer(max_features=50000,ngram_range=(1,2)) #max+features 50k with uni and bigrams like "good" "boy" = "good boy" both like this  even single good and with good boy are taken and top n features
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

clf=LogisticRegression(max_iter=500,class_weight="balanced")
clf.fit(X_train_tfidf, y_train)

print(classification_report(y_test, clf.predict(X_test_tfidf)))

              precision    recall  f1-score   support

    POLITICS       0.98      0.95      0.97      7121
      SPORTS       0.85      0.91      0.88      1015
    WELLNESS       0.93      0.97      0.95      3589

    accuracy                           0.95     11725
   macro avg       0.92      0.94      0.93     11725
weighted avg       0.96      0.95      0.95     11725



## Summarization with BART

In [12]:
from transformers import pipeline
summarizer=pipeline("summarization", model="facebook/bart-large-cnn") #pipline is a huggingface pretrained model

def predict(text):
    clean=preprocess(list(nlp.pipe([text]))[0])
    category=clf.predict(tfidf.transform([clean]))[0]
    confidence=clf.predict_proba(tfidf.transform([clean])).max()
    summary=summarizer(text[:1024],max_length=120,min_length=30,do_sample=False)[0]['summary_text'] #used greedy decoding instead of sampling hich may have different output each time with low proba too!...sumarry_text is used in hugging face to get actual text iterm one [0]
    return category, round(confidence, 3),summary

article="President Biden announced a new technology investment plan worth $5 billion..."
category, confidence, summary = predict(article)

print(f"Category  : {category}")
print(f"Confidence: {confidence}")
print(f"Summary   : {summary}")

Your max_length is set to 120, but your input_length is only 15. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)


Category  : POLITICS
Confidence: 0.888
Summary   : President Biden announced a new technology investment plan worth $5 billion. The plan is expected to be implemented by the end of the year. The White House says the plan will focus on technology and innovation.


In [ ]:
'''
1.CLEAN
nlp.pipe([text]) — runs spaCy on your text, note [text] wraps it in a list because pipe expects a list
list(...)[0] — converts the generator to a list and takes the first (only) item
preprocess(...) — applies your preprocessing function to get clean text
Result is a clean string like "biden announce technology investment boost"

2.CONFIDENCE
tfidf.transform([clean]) — converts clean text to TF-IDF numbers the model understands
clf.predict(...) — runs logistic regression to predict the category
[0] — gets the first (only) result from the array
Result is a string like "POLITICS" or "SPORTS"

3.CATEGORY
predict_proba — instead of just the category, gives probability for each category
For example → [0.02, 0.91, 0.04, 0.03] meaning 91% POLITICS
.max() — gets the highest probability which is the confidence score
Result is a number like 0.91
'''